# Imports

In [1]:
from pathlib import Path
import torch

# Generations

In [2]:
base_path = Path(
    "/projects/static2dynamic/Gravier/schron/experiments_debug/biotine_7d_unet_image128_02/inference_28-11_14-04"
)
assert base_path.exists(), f"Base path {base_path} does not exist."

In [3]:
all_files = list(base_path.rglob("*.pt"))
print(f"Found {len(all_files)} .pt files.")

Found 384 .pt files.


In [4]:
shapes = {}

for file in all_files[:4]:
    d = torch.load(file, mmap=True)
    assert list(d.keys()) == ["generated", "time", "batch_index"], (
        f"Unexpected keys in {file}: {d.keys()}"
    )
    assert len(d["time"]) == d["generated"].shape[0], (
        f"Mismatch in lengths in {file}: len(time)={len(d['time'])}, generated.shape[0]={d['generated'].shape[0]}"
    )
    print(d["generated"].shape, len(d["time"]))

torch.Size([602, 20, 3, 128, 128]) 602
torch.Size([6, 20, 3, 128, 128]) 6
torch.Size([6, 20, 3, 128, 128]) 6
torch.Size([6, 20, 3, 128, 128]) 6


# Save generated images

In [ ]:
from torchvision.utils import save_image
from tqdm import tqdm
import numpy as np


def denormalize(tensor):
    """
    Denormalize from [-1, 1] back to [0, 1] range.
    Original transform: Normalize(mean=[0.5]*3, std=[0.5]*3)
    This means: normalized = (x - 0.5) / 0.5 = 2*x - 1
    So to reverse: x = (normalized + 1) / 2
    """
    return (tensor + 1) / 2

In [ ]:
# Filter files with exactly 6 times (exclude the one with 602 frames)
valid_files = []
for file in tqdm(all_files):
    d = torch.load(file, mmap=True)
    if len(d["time"]) == 6:
        valid_files.append(file)
    else:
        print(f"Skipping {file.name}: has {len(d['time'])} frames instead of 6")

print(f"\nFound {len(valid_files)} valid files with 6 time steps.")

In [ ]:
# Create output directory structure
images_dir = (
    Path(
        "/projects/static2dynamic/Thomas/experiments/schron/biotine_7d_unet_image128_02/inference_28-11_14-04"
    )
    / "images"
)
images_dir.mkdir()

In [ ]:
# Create time subdirectories
for i in range(1, 7):
    time_dir = images_dir / f"time{i}"
    time_dir.mkdir()

# Save images for each file and time step
for file_idx, file in enumerate(tqdm(valid_files, desc="Processing files")):
    d = torch.load(file)
    generated = d["generated"]  # Shape: [6, batch_size, C, H, W] or similar
    times = d["time"]
    batch_idx = d["batch_index"]

    # Iterate over each time step
    for time_idx in range(len(times)):
        time_dir = images_dir / f"time{time_idx + 1}"

        # Get images for this time step
        # generated shape is [n_times, ...] where ... could be [C, H, W] per image
        img_data = generated[time_idx]

        # Denormalize
        img_data = denormalize(img_data)

        # Clamp to valid range
        img_data = torch.clamp(img_data, 0, 1)

        # If it's a batch of images, save each one
        if img_data.dim() == 4:  # [B, C, H, W]
            for img_idx in range(img_data.shape[0]):
                img_name = f"file{file_idx:04d}_batch{batch_idx}_img{img_idx:04d}.png"
                save_image(img_data[img_idx], time_dir / img_name)
        elif img_data.dim() == 3:  # [C, H, W] single image
            img_name = f"file{file_idx:04d}_batch{batch_idx}.png"
            save_image(img_data, time_dir / img_name)
        else:
            print(f"Unexpected tensor shape: {img_data.shape}")

print(f"\nImages saved to {images_dir}")

# FID Computation

In [ ]:
# Install pytorch-fid if needed
# !pip install pytorch-fid

from torchmetrics.image.fid import FrechetInceptionDistance
from torchvision.io import read_image
import pandas as pd

In [ ]:
# Set path to the real/ground truth images
real_images_base_path = Path(
    "/projects/static2dynamic/datasets/biotine/3_channels_min_99_perc_normalized_rgb_stacks_png/patches_255"
)

# Mapping between generated times and real times
# Original selection: times = list(range(0, 19, 3)) = [0, 3, 6, 9, 12, 15, 18] (0-indexed)
# First frame (index 0) is the input/conditioning frame, so 6 generated times are:
# gen_time 1 -> real index 3 -> folder "4" (1-indexed folders)
# gen_time 2 -> real index 6 -> folder "7"
# gen_time 3 -> real index 9 -> folder "10"
# gen_time 4 -> real index 12 -> folder "13"
# gen_time 5 -> real index 15 -> folder "16"
# gen_time 6 -> real index 18 -> folder "19"

gen_to_real_folder = {
    1: "4",  # gen time 1 -> real folder 4
    2: "7",  # gen time 2 -> real folder 7
    3: "10",  # gen time 3 -> real folder 10
    4: "13",  # gen time 4 -> real folder 13
    5: "16",  # gen time 5 -> real folder 16
    6: "19",  # gen time 6 -> real folder 19
}

print(f"Real images base path: {real_images_base_path}")
print(f"Exists: {real_images_base_path.exists()}")
print(f"\nTime mapping (gen -> real folder):")
for gen_t, real_f in gen_to_real_folder.items():
    real_path = real_images_base_path / real_f
    exists = "✓" if real_path.exists() else "✗"
    print(f"  time{gen_t} -> folder {real_f} {exists}")

In [ ]:
def load_images_from_dir(img_dir: Path, max_images: int = None) -> torch.Tensor:
    """Load all images from a directory and return as a tensor."""
    img_files = list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpg"))
    if max_images:
        img_files = img_files[:max_images]

    images = []
    for img_file in tqdm(img_files, desc=f"Loading {img_dir.name}", leave=False):
        img = read_image(str(img_file))  # Returns [C, H, W] in uint8 [0, 255]
        images.append(img)

    return torch.stack(images)  # [N, C, H, W]


def compute_fid(
    real_images: torch.Tensor,
    gen_images: torch.Tensor,
    device="cuda:0",
    batch_size: int = 64,
) -> float:
    """
    Compute FID between real and generated images.
    Both inputs should be [N, C, H, W] tensors in uint8 [0, 255] range.
    """
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)

    # FID expects images in [0, 255] uint8 format
    # If images are float [0, 1], convert them
    if real_images.dtype == torch.float32:
        real_images = (real_images * 255).to(torch.uint8)
    if gen_images.dtype == torch.float32:
        gen_images = (gen_images * 255).to(torch.uint8)

    # Update with real images
    for i in range(0, len(real_images), batch_size):
        batch = real_images[i : i + batch_size].to(device)
        fid.update(batch, real=True)

    # Update with generated images
    for i in range(0, len(gen_images), batch_size):
        batch = gen_images[i : i + batch_size].to(device)
        fid.update(batch, real=False)

    return fid.compute().item()

In [ ]:
# Compute FID for each time step: generated time vs real time
fid_results = {}
device = "cuda:0"
fid_batch_size = 64  # <-- Change this to adjust batch size for FID computation
print(f"Using device: {device}")
print(f"FID batch size: {fid_batch_size}")

for gen_time_idx, real_folder in gen_to_real_folder.items():
    gen_dir = images_dir / f"time{gen_time_idx}"
    real_dir = real_images_base_path / real_folder

    if not real_dir.exists():
        print(
            f"WARNING: Real images directory {real_dir} does not exist, skipping time{gen_time_idx}"
        )
        continue

    print(f"\nComputing FID for gen_time{gen_time_idx} vs real_folder {real_folder}...")

    # Load images
    gen_images = load_images_from_dir(gen_dir)
    real_images = load_images_from_dir(real_dir)

    print(f"  Generated images: {gen_images.shape}")
    print(f"  Real images: {real_images.shape}")
    # Compute FID
    fid_score = compute_fid(
        real_images, gen_images, device=device, batch_size=fid_batch_size
    )
    fid_results[f"time{gen_time_idx} (real:{real_folder})"] = fid_score
    print(f"  FID: {fid_score:.4f}")

print("\n" + "=" * 50)
print("FID Results per Time Step:")
for time_name, fid_val in fid_results.items():
    print(f"  {time_name}: {fid_val:.4f}")

In [ ]:
# Compute FID for ALL times combined
print("\nComputing FID for all times combined...")

# Collect all generated images
all_gen_images = []
all_real_images = []

for gen_time_idx, real_folder in gen_to_real_folder.items():
    gen_dir = images_dir / f"time{gen_time_idx}"
    real_dir = real_images_base_path / real_folder

    if not real_dir.exists():
        print(
            f"WARNING: Skipping time{gen_time_idx} - real dir {real_folder} does not exist"
        )
        continue

    gen_images = load_images_from_dir(gen_dir)
    real_images = load_images_from_dir(real_dir)

    all_gen_images.append(gen_images)
    all_real_images.append(real_images)

all_gen_images = torch.cat(all_gen_images, dim=0)
all_real_images = torch.cat(all_real_images, dim=0)

print(f"Total generated images: {all_gen_images.shape}")
print(f"Total real images: {all_real_images.shape}")

fid_all = compute_fid(
    all_real_images, all_gen_images, device=device, batch_size=fid_batch_size
)
fid_results["all_times"] = fid_all
print(f"FID (all times): {fid_all:.4f}")

In [ ]:
# Summary table
print("\n" + "=" * 50)
print("FINAL FID RESULTS")
print("=" * 50)

results_df = pd.DataFrame([{"Time": k, "FID": v} for k, v in fid_results.items()])
print(results_df.to_string(index=False))

# Save results
results_path = base_path / "fid_results.csv"
results_df.to_csv(results_path, index=False)
print(f"\nResults saved to {results_path}")